# Week 6 Homework: Your Answers vs. Your Vibe-Coded Dashboard

**Week 4 dashboard link:** _(paste your Streamlit URL)_

---

Two weeks ago you vibe-coded a MovieLens dashboard before you had the tools to judge it. Since then you've had the stats lecture (Week 5) and this week's EDA lecture. Now you do it the real way: answer the same 4 questions yourself, by hand, then audit the dashboard against your own work.

**Ground rules**

- 🚫 **No AI for code or answers in Parts 1 and 2.** Lecture notebooks, pandas/seaborn docs, Google, and Stack Overflow are all fair game. Write Part 3 in your own words too.
- 🔒 **Lock in before you peek.** Finish Parts 1 and 2 (answers written down) *before* you open your dashboard. If you look first, you'll anchor on its numbers and the comparison is worthless.
- 🔢 **Show your numbers.** Every answer needs actual values (counts, averages, titles), not just a vibe.
- ✏️ Written answers go in markdown cells (double-click to edit, Shift+Enter to render). Most are 2 to 4 sentences. Short and specific beats long and vague.

## The data

`movie_ratings.csv` has one row per rating: 100,000 ratings from 943 users on 1,682 movies.

- `user_id`, `movie_id`: who rated, which movie
- `rating`: 1 to 5 stars
- `timestamp`, `rating_year`: when the rating was given
- `age`, `gender`, `occupation`, `zip_code`: the rater
- `title`: movie title, with release year in parentheses
- `year`, `decade`: the movie's release year and decade
- `genres`: all of the movie's genres in one string, separated by the pipe character (a movie can have several)

## Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ["data/movie_ratings.csv", "../data/movie_ratings.csv"] if Path(p).exists())
df = pd.read_csv(data_path)
df.head()

,user_id,movie_id,rating,timestamp,age,gender,occupation,zip_code,title,year,decade,genres,rating_year
0,196,242,3,1997-12-04 15:55:49,49,M,writer,55105,Kolya (1996),1996.0,1990.0,Comedy,1997
1,186,302,3,1998-04-04 19:22:22,39,F,executive,00000,L.A. Confidential (1997),1997.0,1990.0,Crime|Film-Noir|Mystery|Thriller,1998
2,22,377,1,1997-11-07 07:18:36,25,M,writer,40206,Heavyweights (1994),1994.0,1990.0,Children|Comedy,1997
3,244,51,2,1997-11-27 05:02:03,28,M,technician,80525,Legends of the Fall (1994),1994.0,1990.0,Drama|Romance|War|Western,1997
4,166,346,1,1998-02-02 05:33:16,47,M,educator,55113,Jackie Brown (1997),1997.0,1990.0,Crime|Drama,1998


---
# Part 1: Methodology

## 1.1 Know your data first

Before answering anything, check what you're working with: shape, column types, missing values, duplicate rows, and the range of the numeric columns.

In [2]:
# Your data checks here

df.shape

(100000, 13)

In [3]:
df.dtypes

user_id          int64
movie_id         int64
rating           int64
timestamp       object
age              int64
gender          object
occupation      object
zip_code        object
title           object
year           float64
decade         float64
genres          object
rating_year      int64
dtype: object

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   user_id      100000 non-null  int64  
 1   movie_id     100000 non-null  int64  
 2   rating       100000 non-null  int64  
 3   timestamp    100000 non-null  object 
 4   age          100000 non-null  int64  
 5   gender       100000 non-null  object 
 6   occupation   100000 non-null  object 
 7   zip_code     100000 non-null  object 
 8   title        100000 non-null  object 
 9   year         99970 non-null   float64
 10  decade       99970 non-null   float64
 11  genres       100000 non-null  object 
 12  rating_year  100000 non-null  int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 9.9+ MB


In [5]:
df.isna().sum()

user_id         0
movie_id        0
rating          0
timestamp       0
age             0
gender          0
occupation      0
zip_code        0
title           0
year           30
decade         30
genres          0
rating_year     0
dtype: int64

In [6]:
df.duplicated().sum()

0

In [7]:
df.describe()

,user_id,movie_id,rating,age,year,decade,rating_year
count,100000.00000,100000.000000,100000.000000,100000.000000,99970.000000,99970.000000,100000.000000
mean,462.48475,425.530130,3.529860,32.969850,1987.748224,1982.780134,1997.471010
std,266.61442,330.798356,1.125674,11.562623,14.140014,13.820687,0.499161
min,1.00000,1.000000,1.000000,7.000000,1922.000000,1920.000000,1997.000000
25%,254.00000,175.000000,3.000000,24.000000,1986.000000,1980.000000,1997.000000
50%,447.00000,322.000000,4.000000,30.000000,1994.000000,1990.000000,1997.000000
75%,682.00000,631.000000,4.000000,40.000000,1996.000000,1990.000000,1998.000000
max,943.00000,1682.000000,5.000000,73.000000,1998.000000,1990.000000,1998.000000


**✏️ Data notes:** _(What did you find? Anything missing, duplicated, or surprising? What did you do about it, and why?)_

In [8]:
df = df.dropna(subset=["year", "decade"])
df.shape

(99970, 13)

Found out we have 30 null values for both years and decade. Also there are no duplicated values. I think for the null values we'll choose to drop them in order to skip out on that data.

## 1.2 Decide before you chart

In Week 4 the questions were ambiguous on purpose. This is where *you* commit to an interpretation. Fill it in now, and update it if you change your mind while working (just say so).

**Q1 Genre breakdown**
- What you have to decide: What am I counting, ratings or movies? How do I handle movies that have several genres?
- My choice: I'll count the movies, and I'll split each movie into its separate genres (i.e movie counts once in every genre it's classified as)
- Why: we're doing movies that were rated, so we're looking at movies. counting ratings could inflate movies that occur more often in movies.csv

**Q2 Genre satisfaction**
- What you have to decide: "Average rating" of what? How do I treat genres that have very little data?
- My choice: We are getting the average rating per genre, We could do what we did in week 4 where we rank genres with at least 50 ratings to prevent little data
- Why: Mostly deciding based on what we did for Week 4

**Q3 Ratings over time**
- What you have to decide: Is every year backed by the same amount of data? What do I do with movies that have no release year?
- My choice: Every year isn't backed by the same amount of data, i think lets choose the same threshold we said earlier where any year that doesn't have 50 ratings we just drop.
- Why: Mostly deciding based on what we did for Week 4

**Q4 Best movies**
- What you have to decide: What is a movie's rating (the mean of what)? Is "at least 50" ≥ or >? What if there's a tie?
- My choice: The movie rating is going to be the mean of all it's ratings with our threshold being at least 50, movies with 50 should count. We can break ties using whichever movie has more ratings, then we could just do it alphabetically
- Why: Movies with more ratings with a tied rating obviously have more weight/meaning to its rating, also idk why leave out movies with 50 ratings that should be included

---
# Part 2: Results (by hand, no AI)

Answer each question with a chart or table **and** a written answer with real numbers. Pick the chart type yourself and be ready to defend it in Part 3. Add as many code cells as you need.

## Q1. Genre breakdown

**What's the distribution of genres among the movies that were rated?**

<details>
<summary>Stuck on movies with several genres? Hint</summary>

`genres` is one string like `Crime|Film-Noir|Mystery|Thriller`. To get one row per genre:

```python
genre_df = df.assign(genre=df['genres'].str.split('|')).explode('genre')
```

Heads up: after this, a movie with 3 genres takes up 3 rows. Think about what that does to anything you count or average. Exploding is fine for "how often does a genre show up", but be careful calling the result "market share" (the percentages add up to more than 100%).
</details>

In [9]:
# Q1: your code here
movies = df.drop_duplicates("movie_id")                       
genre_df = movies.assign(genre=movies["genres"].str.split("|")).explode("genre")
counts = genre_df["genre"].value_counts()
share = (counts / len(movies) * 100).round(1)                 
pd.DataFrame({"movies": counts, "pct_of_movies": share})


,movies,pct_of_movies
genre,,
Drama,724,43.2
Comedy,505,30.1
Thriller,251,15.0
Action,251,15.0
Romance,247,14.7
Adventure,135,8.1
Children,121,7.2
Crime,109,6.5
Sci-Fi,101,6.0


**✏️ My answer (with numbers):** _(Which genres dominate, by how much, and what exactly did you count?)_

Drama, Comedy, and Thriller make up almost 90%

## Q2. Genre satisfaction

**Which genres have the highest average rating? Which have the lowest?**

Show how much data is behind each average (for example, the count next to it). An average of 10 ratings and an average of 10,000 ratings are not the same kind of evidence.

In [10]:
# Q2: your code here
rated = df.assign(genre=df["genres"].str.split("|")).explode("genre")
stats = (rated.groupby("genre")["rating"]
              .agg(mean="mean", num_ratings="size")
              .round(2)
              .sort_values("mean", ascending=False))
stats


,mean,num_ratings
genre,,
Film-Noir,3.92,1733
War,3.82,9398
Documentary,3.69,744
Drama,3.69,39894
Mystery,3.64,5245
Crime,3.63,8055
Romance,3.62,19461
Western,3.61,1854
Animation,3.58,3599


**✏️ My answer (with numbers):** _(Top and bottom genres, their averages, and how much data each one rests on.)_

## Q3. Ratings over time

**How has the mean rating changed across movie release years?**

Show how many ratings each year is based on, alongside the trend.

In [11]:
# Q3: your code here
by_year = (df.dropna(subset=["year"])
             .groupby("year")["rating"]
             .agg(mean="mean", num_ratings="size")
             .round(2))
by_year[by_year.num_ratings >= 50]


,mean,num_ratings
year,,
1922.0,3.56,54
1933.0,3.97,94
1934.0,4.02,161
1935.0,3.87,127
1937.0,3.75,224
...,...,...
1994.0,3.42,9034
1995.0,3.41,12389
1996.0,3.31,18744


**✏️ My answer (with numbers):** _(What's the trend? Is it equally believable across the whole timeline?)_

## Q4. Best movies, with a floor

**What are the 5 best-rated movies that have at least 50 ratings? What changes if you raise the floor to 150?**

In [12]:
# Q4: your code here
movie_stats = (df.groupby(["movie_id", "title"])["rating"]
                 .agg(mean="mean", n_ratings="size")
                 .reset_index())

fifty_floor = (movie_stats[movie_stats.n_ratings >= 50]
               .sort_values(["mean", "n_ratings", "title"], ascending=[False, False, True])
               .head(5))

one_fifty_floor = (movie_stats[movie_stats.n_ratings >= 150]
                   .sort_values(["mean", "n_ratings", "title"], ascending=[False, False, True])
                   .head(5))

fifty_floor, one_fifty_floor

(     movie_id                                              title      mean  \
 406       408                              Close Shave, A (1995)  4.491071   
 316       318                            Schindler's List (1993)  4.466443   
 168       169                         Wrong Trousers, The (1993)  4.466102   
 481       483                                  Casablanca (1942)  4.456790   
 113       114  Wallace & Gromit: The Best of Aardman Animatio...  4.447761   
 
      n_ratings  
 406        112  
 316        298  
 168        118  
 481        243  
 113         67  ,
      movie_id                             title      mean  n_ratings
 316       318           Schindler's List (1993)  4.466443        298
 481       483                 Casablanca (1942)  4.456790        243
 63         64  Shawshank Redemption, The (1994)  4.445230        283
 601       603                Rear Window (1954)  4.387560        209
 11         12        Usual Suspects, The (1995)  4.385768       

**✏️ My answer (with numbers):** _(Both top-5 lists. Which movies dropped out or moved in when the floor went up, and what does that say about small samples?)_

> 🔒 **Stop here.** Parts 1 and 2 should be completely filled in before you open your dashboard.

---
# Part 3: Analysis

Now open your Week 4 dashboard.

## 3.1 How the dashboard got built

Paste 2 prompts that shaped it (from your build log, or from your AI chat history if you didn't keep a log; don't invent any). For each one: what did you ask, what did the AI give you first, and what did you change or accept without checking?

**Prompt 1**
- What I asked: > Genre Breakdown: Distribution of genres among movies that were rated? (Movies can have multiple genres- Please explain how it handled that before you count anything)

> I would assume some kind of Pie Chart would be used for this, let me know if something better can be used

> I think a bar chart would likely be the best use for this given we are comparing averages across categories, and we would want it to be sorted

> We would need some kind of filtered bar chart i guess and a bar to filter through floors from 50-150
- AI's first attempt: I recommend sorted horizontal bars for both genre charts, a line chart for release years, and a ranked horizontal bar chart with a minimum-ratings slider for the top five.
- What I changed (or accepted without checking): Accepted without checking (lol)

**Prompt 2**
- What I asked: One-shotted the prompt
- AI's first attempt:
- What I changed (or accepted without checking):

## 3.2 Compare the answers

Read the answer off your dashboard for each question and put it next to yours.

**Q1**
- Dashboard says: Drama 725 movies (43.1%), Comedy 505 (30.0%), Action and Thriller 251 each (14.9%), Romance 247 (14.7%). Counts distinct movies, one movie counted once per genre, out of all 1,682 rated movies.
- I say: Drama 724 (43.2%), Comedy 505 (30.1%), Thriller and Action 251 each (15.0%), Romance 247 (14.7%).
- Same? (yes / partly / no): partly (same ranking, a few counts off by 1 or 2)
- If different, why? See below.

**Q2**
- Dashboard says: highest Film-Noir 3.92 (1,733 ratings, 24 movies), then War 3.82 and Drama 3.69. Lowest Fantasy 3.22 (1,352 ratings); the verdict box skips "unknown" (3.20, 10 ratings) but the chart still shows it.
- I say: highest Film-Noir 3.92 (1,733), then War 3.82. Lowest Fantasy 3.22 (1,352), but my table also has "unknown" at 1.00 from a single rating, since I never applied the 50-rating floor I planned in 1.2.
- Same? (yes / partly / no): partly (same top and bottom; Documentary 3.67 vs my 3.69 swaps places with Drama, and "unknown" differs)
- If different, why? Documentary and "unknown" lost ratings in my version (the 30 dropped rows), and I never filtered small genres.

**Q3**
- Dashboard says: line over all 71 release years (1922 to 1998). Mean is about 3.8 to 4.0 for the 1930s and 1940s and slides to about 3.3 by 1996 to 1998 (1996: 3.31 on 18,744 ratings). Hover shows sample sizes, but tiny years like 1926 (2 ratings) are plotted at full weight.
- I say: same trend over 65 years, because I kept only years with at least 50 ratings (1934: 4.03 on 161 ratings; 1996: 3.31; 1997: 3.31; 1998: 3.30 on 851).
- Same? (yes / partly / no): partly (identical numbers for every shared year, different set of years)
- If different, why? See below.

**Q4**
- Dashboard says: floor 50: Close Shave, A (4.49, 112 ratings), Schindler's List (4.47, 298), Wrong Trousers, The (4.47, 118), Casablanca (4.46, 243), Wallace & Gromit: The Best of Aardman Animation (4.45, 67). Floor 150: Schindler's List, Casablanca, Shawshank Redemption, Rear Window (4.39, 209), Usual Suspects (4.39, 267). Dashboard uses >=.
- I say: identical lists, order and means, at both floors.
- Same? (yes / partly / no): yes
- If different, why? n/a. I trust it because both versions use the mean of all ratings per movie, >= on the floor, and no tie occurs in the top 5 so the tiebreak rules (count then movie ID vs count then title) never matter. Going from 50 to 150 drops the three 67 to 118-rating movies (Close Shave, Wrong Trousers, Wallace & Gromit) and brings in Shawshank, Rear Window and Usual Suspects, so the small-sample movies were riding on fewer ratings.

**When numbers differ, check these first.** Most mismatches come from one of these:

- Counting **ratings** vs. counting **movies**
- One row per rating vs. **one row per rating per genre** (after exploding)
- **Mean of all ratings** vs. the mean of each movie's average
- "At least 50" as ≥ vs. >, or applying the floor before vs. after another step
- What happened to movies with a **missing year**
- Genres with **tiny sample sizes** left in or filtered out

**✏️ Pick two questions and explain what's going on** (2 to 3 sentences each). Track down the *actual cause*: read your dashboard's code, or re-run it with one change. "The AI did it differently" is not an explanation. If the answers match, say why you now trust them.


**Q1 explanation:** The dashboard keeps the 5 movies with a missing release year; my `dropna(subset=["year","decade"])` in 1.1 removed their 30 ratings before I counted anything. That is exactly why Drama is 725 vs 724, Children 122 vs 121, Animation 42 vs 41, Documentary 50 vs 48 and "unknown" 2 vs 1: the counting method (distinct movies, exploded genres) is the same, only the movie set differs. I also should not read "Drama, Comedy, Thriller almost 90%" as a share, since those percentages overlap and are not additive.

**Q3 explanation:** Both use the mean of all ratings per release year, so every year they share matches to the decimal. The difference is my 50-rating cutoff, which removed 1926, 1930, 1931, 1932, 1936 and 1943 (some with only 2 to 44 ratings), while the dashboard plots them and its line zig-zags in the early decades because those years rest on one or two movies. So the trend is only believable from the 1930s on where there is more data, and especially the 1990s where thousands of ratings back each year.

## 3.3 Chart choices

**Q1**
- Chart the dashboard used: Horizontal Bar
- Chart I used: None
- Which communicates the answer better, and why?

**Q2**
- Chart the dashboard used: Horizontal Bar
- Chart I used: None
- Which communicates the answer better, and why?

**Q3**
- Chart the dashboard used: Line with markers
- Chart I used: None
- Which communicates the answer better, and why?

**Q4**
- Chart the dashboard used: Horizontal Bar
- Chart I used: None 
- Which communicates the answer better, and why?

Use what you learned about chart choice and misleading charts (Week 5) and this week's lecture. Is it sorted? Are the labels readable? Are the axes honest? Does it show how much data is behind each number?

## 3.4 What can this data actually tell you?

**Answer at least 3 of these 5**, using your own results as evidence.

**1. Who is in this data?** Check the fewest and the most ratings any single user has given (`df.groupby('user_id').size()`), and look at `rating_year`. What kind of person ends up in a dataset like this, and who is missing? How might that tilt your Q2 answer? *(Week 5: sampling bias)*

**2. How sure are you about the bottom genre?** Take the lowest-rated genre from your Q2. How many ratings *and* how many distinct movies is it built on? Estimate a rough 95% interval: mean ± 1.96 × std / √n. Would you bet it's truly the worst genre? *(Week 5: standard error and confidence intervals. Also ask yourself: are 5,000 ratings of the same handful of movies really 5,000 independent pieces of evidence? Remember the one-row-per-model lesson from last week's homework.)*

**3. What can Q3 really say about time?** How many different values does `rating_year` have? So can this data tell you whether people's taste changed over time? What does your Q3 trend actually describe instead? Why might an old movie in this dataset score higher than a recent one? *(Think: who rates a 1940s film in 1997, and which old movies are even in the data?)*

**4. Does "average" depend on how you average?** Recompute Q2 two ways: (a) average over all ratings, (b) average each movie first, then average the movies within each genre. Does the top genre change? "Which genre do *people* like most?" and "which genre has the best *movies*?" are different questions. *(Week 5: confounding, Simpson's paradox)*

**5. Deployment risk.** A studio exec uses your dashboard to decide which genre to green-light. What's the most misleading conclusion they could draw from one of your charts? Write the one-line caveat you'd put under that chart.

**✏️ Your answers:** _(label each one with its number)_

## 3.5 Verdict

**Where I'd override the AI** (something it got wrong, hid, or did poorly, and what your version did better): Cleaning up the Data when it comes to null values, duplicates, or setting boundaries, etc.

**Where the AI held up** (something it did well, or a choice you doubted at first that turned out right): AI did well at determining what charts would best represent the data.

**One sentence:** looking back at both weeks, who was driving, you or the AI, and when? AI drove a lot of the syntax help most of the time, but when it came to decision making I wanted to make sure I was the one at the fore front

---
## Optional stretch

Only if you want more practice:

- Pick 4 genres. How does the mean rating change as viewer age increases? (Try more than 4 and see which genres make the most interesting chart.)
- Plot number of ratings vs. mean rating for each movie. Do the most-rated movies get higher ratings? Is there a correlation? *(Week 5: correlation vs. causation)*

---
## Before you submit

- [ ] Copy this notebook into the `exercise/` folder and rename it `Firstname_Lastname_week_06_exercise.ipynb`
- [ ] **Kernel → Restart & Run All** and make sure it runs top to bottom with your charts showing
- [ ] Every ✏️ cell and every fill-in line is filled in
- [ ] Open a PR from your fork, same as every other week